In [ ]:
from option_analyzer import *
self = OptionAnalyzer('quotes', 'chain')

In [ ]:
symlist = self.get_updated_symbol_list(age_ub=60)
df_quotes, df_shortint, df_vola = self.get_quote_df(symlist)
df_earning = self.count_days_from_earning_reports(df_quotes)
if df_earning.shape[0] == 0:
    d2e = {}
else:
    d2e = df_earning['earningDays'].to_dict()
print('Days to E:', d2e)
df_raw = self.build_option_df(symlist)
px.bar(self.check_data_age(df_raw), barmode='group', width=60*len(symlist), height=300).show()

In [ ]:
df_call = self.select_options_by_type(df_raw, 'call')
self.plot_option_stats(df_call)
pp = ParallelOptionCalculator(df_call, self, f'/run/user/{os.getuid()}/time_decay_call')
csv_files = pp.do_all_theta_curves(symlist)
dfc = pp.assemble_time_decay_df(csv_files, d2e)
print('hdte_resid check:', dfc[(dfc.hdte_resid - dfc.resid) <= -1e-6].shape)

### Call Options: ignore no-bid or low open interest (minimum open interests is 100)

In [ ]:
hdte_resid_lb = 0.95
overpaid_ub = 0.1
spread_ub = 5
_filter = (dfc.dte >= 90) & (dfc.moneyness <= 1.0) & (dfc.hdte_resid >= hdte_resid_lb) & (dfc.overpaid <= overpaid_ub) & (dfc.pctSpread <= spread_ub)
_filter = _filter  & (dfc.symbol != 'TLT')
_dfc = dfc[_filter].drop(columns=['dth', 'dtz', 'dthr', 'dtzr']).sort_values(by='leverage', ascending=False)
_dfc.head(20)

### Top leverage

In [ ]:
_filter = (dfc.pctSpread <= 5) & (dfc.moneyness <= 1) & (dfc.dte >= 60) & (dfc.symbol != 'TLT')
px.scatter(dfc[_filter].sort_values(by='leverage', ascending=False).head(1000), x='hdte_resid', y='leverage', color='symbol', height=600)

In [ ]:
px.scatter(dfc[(dfc.symbol=='SPY') & (dfc.expDt == '2026-06-18') & (dfc.hdte_resid >= 0.9)], x='strike', y='leverage', color='hdte_resid', height=800)

In [ ]:
px.scatter(dfc[dfc.symbol.str.contains('QQQ|SPY') & (dfc.dte >= 90) & (dfc.dte <= 300) & (dfc.moneyness <= 1) & (dfc.hdte_resid >= 0.9) & (dfc.hdte_resid <= 0.99)], x='hdte_resid', y='leverage', color='expDt', height=800)

### The End